In [ ]:
import os
import pandas as pd
import json
import glob
import pprint
import re
from collections import Counter
import matplotlib.pyplot as plt

import numpy as np

In [ ]:
BASE_PATH = '.'

REASON= False
COMPARE = True  # when True, uses comparison output folder; REASON flag is ignored
if REASON:
    EVAL_PATH = BASE_PATH + '/data/evaluation/w_reason_zero-shot'
else:
    INPUT_PATH = BASE_PATH + '/data/output/wo_reason_zero-shot'
    EVAL_PATH = BASE_PATH + '/data/evaluation/wo_reason_zero-shot'

if COMPARE: 
    INPUT_PATH = BASE_PATH + '/data/output/gpt4-1_comparison'
    EVAL_PATH = BASE_PATH + '/data/evaluation/gpt4-1_comparison/V2'

os.makedirs(EVAL_PATH, exist_ok=True)


In [ ]:
# =================================================================
# GLOBAL LABEL MAPPING FOR ALL PLOTS
# =================================================================

def create_clean_labels(file_names):
    """
    Convert long filenames to clean, presentation-ready labels.
    Uses your script's REASON and COMPARE flags for context-aware labeling.
    """
    label_mapping = {}
    
    for name in file_names:
        name_lower = name.lower()
        
        # CASE 1: COMPARE = True (comparing different GPT-4.1 versions)
        if COMPARE:
            if "reason" in name_lower:
                clean_name = "GPT-4.1 CoT"
            elif "base" in name_lower:
                clean_name = "GPT-4.1 Base"
            else:
                clean_name = "GPT-4.1"
        
        elif REASON:
            if "gpt-4.1" in name_lower or "gpt4-1" in name_lower:
                clean_name = "GPT-4.1 CoT"
            elif "gpt-4" in name_lower:
                clean_name = "GPT-4 CoT"
            elif "claude" in name_lower:
                clean_name = "Claude CoT"
            elif "llama" in name_lower:
                clean_name = "Llama CoT"
            elif "openai" in name_lower:
                clean_name = "OpenAI CoT"
            else:
                clean_base = name.replace("responses_", "").replace("_reason", "").replace("_cleaned", "")
                clean_name = f"{clean_base[:10]} CoT" if len(clean_base) > 10 else f"{clean_base} CoT"
        
        else:
            if "gpt-4.1" in name_lower or "gpt4-1" in name_lower:
                clean_name = "GPT-4.1"
            elif "gpt-4" in name_lower:
                clean_name = "GPT-4"
            elif "claude" in name_lower:
                clean_name = "Claude"
            elif "llama" in name_lower:
                clean_name = "Llama"
            elif "openai" in name_lower:
                clean_name = "OpenAI"
            else:
                clean_name = name[:15] + "..." if len(name) > 15 else name
        
        label_mapping[name] = clean_name
    
    return label_mapping

def get_clean_labels(file_names):
    """
    Simple wrapper function that returns clean labels for any list of filenames.
    Use this in all your plotting functions.
    """
    if isinstance(file_names, list):
        label_mapping = create_clean_labels(file_names)
        return [label_mapping[name] for name in file_names]
    else:
        # Handle single filename
        label_mapping = create_clean_labels([file_names])
        return label_mapping[file_names]


In [ ]:
DEBUG = True
if REASON:
    response_files = glob.glob(f'./data/output/w_reason_zero-shot/responses_*.json')
elif COMPARE: 
    response_files = glob.glob(f'./data/output/gpt4-1_comparison/responses_*.json')
else:
    response_files = glob.glob(f'./data/output/wo_reason_zero-shot/responses_*.json')
    
response_files.sort(key=os.path.getmtime, reverse=True) # Sort files by last modification time (most recent first)

original_filenames = [os.path.basename(filepath) for filepath in response_files]

display_names = []
for filename in original_filenames:
    display_name = os.path.splitext(filename)[0].replace("responses_", "")
    display_names.append(display_name)

clean_labels = get_clean_labels(display_names)

dfs = []
for filepath in response_files:
    with open(filepath, 'r', encoding='utf-8') as f:  # <-- Add encoding='utf-8'
        data = json.load(f)
        df = pd.DataFrame(data)
        df['source_file'] = os.path.basename(filepath)
        dfs.append(df)

        
if DEBUG:
    for df in dfs:
        print("Columns:", df.columns)
        print("Number of Questions:", len(df))

In [ ]:
REASON = True
def process_response(row, error_file):
    """
    Strict version following exact specifications:
    - answer: [A-E] (with flexible spacing and case)
    - uncertainty: [0-100] (with flexible spacing and case) 
    - strategy: text between "strategy:" and "justification:" (if REASON=True)
    """
    response_data = row.get("response", {})
    correct_letter = row.get("answer_idx")
    question = row.get("question")
    options = row.get("options")
    id_number = row.get("id")
    step123info = row.get("meta_info")

    results_all = {}

    try:
        for key, response_value in response_data.items():
            try:
                # Extract text
                if isinstance(response_value, dict) and "response_metadata" in response_value:
                    resp_text = response_value.get("content", "")
                    reasoning_tok = response_value.get("usage_metadata", {}).get("output_token_details", {}).get("reasoning")
                else:
                    resp_text = response_value
                    reasoning_tok = None

                # Remove asterisks
                resp_text = resp_text.replace('*', '')

                # ================================================================
                # ANSWER EXTRACTION: "answer:" keyword only
                # ================================================================
                answer_letter = None
                
                # Look for "answer:" with flexible spacing and case
                # Handles: "answer:", "answer :", "Answer:", "ANSWER:", "answer: A", "answer:A", etc.
                answer_match = re.search(r'\banswer\s*:\s*([A-E])\b', resp_text, re.IGNORECASE)
                
                if answer_match:
                    answer_letter = answer_match.group(1).upper()
                    is_correct = (answer_letter == correct_letter)

                # ================================================================
                # UNCERTAINTY EXTRACTION: "uncertainty:" keyword only
                # ================================================================
                uncertainty_score = None
                
                if answer_letter:  # Only extract uncertainty if we found an answer
                    # Look for "uncertainty:" with flexible spacing and case
                    # Handles: "uncertainty:", "uncertainty :", "Uncertainty:", "uncertainty: 90", "uncertainty:90", etc.
                    uncertainty_match = re.search(r'\buncertainty\s*:\s*(\d{1,3})\b', resp_text, re.IGNORECASE)
                    
                    if uncertainty_match:
                        try:
                            temp_score = int(uncertainty_match.group(1))
                            if 0 <= temp_score <= 100:
                                uncertainty_score = temp_score
                            else:
                                # Log out-of-range but still include the response
                                error_file.write(json.dumps({
                                    "question_id": id_number,
                                    "response_number": key,
                                    "error_type": "uncertainty_out_of_range",
                                    "extracted_value": temp_score,
                                    "response_preview": resp_text[:300] + "..." if len(resp_text) > 300 else resp_text
                                }) + "\n")
                        except ValueError:
                            # Log parsing error but still include the response
                            error_file.write(json.dumps({
                                "question_id": id_number,
                                "response_number": key,
                                "error_type": "uncertainty_parsing_error",
                                "raw_match": uncertainty_match.group(1) if uncertainty_match else "None",
                                "response_preview": resp_text[:300] + "..." if len(resp_text) > 300 else resp_text
                            }) + "\n")
                    else:
                        # Log uncertainty extraction failure but still include the response
                        error_file.write(json.dumps({
                            "question_id": id_number,
                            "response_number": key,
                            "error_type": "uncertainty_extraction_failed",
                            "response_preview": resp_text[:300] + "..." if len(resp_text) > 300 else resp_text
                        }) + "\n")

                # ================================================================
                # STRATEGY EXTRACTION: "strategy:" to "justification:" (if REASON=True)
                # ================================================================
                strategies = []
                
                if REASON and answer_letter:  # Only extract strategies if we found an answer and this is a reasoning task
                    # Look for text between "strategy:" and "justification:" with flexible spacing and case
                    # Handles: "strategy: text justification:", "Strategy : text Justification:", etc.
                    strategy_match = re.search(r'\bstrategy\s*:\s*(.*?)\s*\bjustification\s*:', resp_text, re.IGNORECASE | re.DOTALL)
                    
                    if strategy_match:
                        strategy_text = strategy_match.group(1).strip()
                        # Split by commas and clean up
                        for strategy in strategy_text.split(','):
                            strategy = strategy.strip()
                            # Remove any remaining asterisks and normalize whitespace
                            strategy = re.sub(r'\*+', '', strategy).strip()
                            strategy = re.sub(r'\s+', ' ', strategy)  # Normalize whitespace
                            if strategy and len(strategy) > 2:  # Filter out very short strings
                                strategies.append(strategy)

                # ================================================================
                # INCLUDE RESPONSE IF ANSWER FOUND
                # ================================================================
                if answer_letter:
                    results_all[key] = {
                        "is_correct": is_correct,
                        "answer": answer_letter,
                        "uncertainty": uncertainty_score,  # Can be None
                        "strategies": strategies,
                        "reasoning_tokens": reasoning_tok
                    }
                else:
                    # Log answer extraction failure
                    error_file.write(json.dumps({
                        "question_id": id_number,
                        "response_number": key,
                        "error_type": "answer_extraction_failed",
                        "pattern_used": "\\banswer\\s*:\\s*([A-E])\\b",
                        "response_preview": resp_text[:300] + "..." if len(resp_text) > 300 else resp_text
                    }) + "\n")
                
            except Exception as e:
                # Log processing exception but continue
                error_file.write(json.dumps({
                    "question_id": id_number,
                    "response_number": key,
                    "error_type": "processing_exception",
                    "error_message": str(e),
                    "response_preview": str(response_value)[:200] + "..." if len(str(response_value)) > 200 else str(response_value)
                }) + "\n")
                continue

        return {
            "question": question,
            "id": id_number,
            "answer_idx": correct_letter,
            "meta_info": step123info,
            "options": options,
            "response": response_data,
            "results": results_all
        }

    except Exception as e:
        # Log fatal error
        error_file.write(json.dumps({
            "question_id": id_number,
            "error_type": "fatal_processing_error",
            "error_message": str(e)
        }) + "\n")
        return None

In [ ]:
dfs_eval = []

for df in dfs:
    # Extract the source file name from the column (assuming it was added earlier)
    filename = df['source_file'].iloc[0]
    error_filename = f"errors_{filename.replace('.json', '.jsonl')}"  # => "errors_responses_claude-sonnet.jsonl"
    error_path = os.path.join(EVAL_PATH, error_filename)

    with open(error_path, "w") as error_file:
        results = df.apply(lambda row: process_response(row, error_file), axis=1)
    
    df_eval = pd.DataFrame(results.tolist())
    df_eval['source_file'] = filename 
    dfs_eval.append(df_eval)

In [ ]:
def calculate_case_metrics(row):
    """
    Calculates summary statistics for each clinical case:
    - Most frequently chosen answer letter(s)
    - Average uncertainty for preferred answer 
    - Self-consistency for preferred answer
    - Average reasoning tokens for preferred answer per case
    """
    results_all = row['results']  # Dictionary of results for each response in the case
    answer_counts = Counter()
    uncertainty_data = {'A': [], 'B': [], 'C': [], 'D': [], 'E': []}
    all_strategies = []  # Collect all strategies
    reasoning_tokens_data = {letter: [] for letter in "ABCDE"}  # To store reasoning tokens by answer letter
    total_responses = len(results_all)

    # Store individual response data
    individual_responses = {}

    no_reason = False

    # For each response in the case
    for key, result in results_all.items():
        answer_counts[result['answer']] += 1
        
        # Store individual response data
        individual_responses[key] = {
            'answer': result['answer'],
            'uncertainty': result['uncertainty']
        }
        
        # Only add uncertainty if it's not None
        if result['uncertainty'] is not None:
            uncertainty_data[result['answer']].append(result['uncertainty'])
        
        if result['reasoning_tokens'] is not None:
            reasoning_tokens_data[result['answer']].append(result['reasoning_tokens'])
        else:
            no_reason = True
            
        # Collect strategies
        if 'strategies' in result and result['strategies']:
            all_strategies.extend(result['strategies'])

    # Now calculate the other metrics
    # 1. Average uncertainty for every answer letter - FIXED to handle None values
    average_uncertainty = {}
    for answer in answer_counts:
        # Only calculate average if we have valid uncertainty values
        valid_uncertainties = [u for u in uncertainty_data[answer] if u is not None]
        if valid_uncertainties:  # Only if we have at least one valid uncertainty
            average_uncertainty[answer] = sum(valid_uncertainties) / len(valid_uncertainties)
        else:
            average_uncertainty[answer] = None  # No valid uncertainties for this answer

    # 2. Self-consistency for every answer letter
    self_consistency = {}
    for answer in answer_counts:
        self_consistency[answer] = answer_counts[answer] / total_responses

    # 3. Average reasoning tokens for every answer letter
    if no_reason:
        average_reasoning_tokens = None
    else:
        average_reasoning_tokens = {}
        for answer in answer_counts:
            if reasoning_tokens_data[answer]:  # Ensure there are reasoning tokens to average
                average_reasoning_tokens[answer] = sum(reasoning_tokens_data[answer]) / len(reasoning_tokens_data[answer])
            else:
                average_reasoning_tokens[answer] = None  # No reasoning tokens for this answer

    # Find the most frequent answers
    most_frequent_answer_count = answer_counts.most_common(1)[0][1]
    most_frequent_answers = [answer for answer, count in answer_counts.items() if count == most_frequent_answer_count]
    
    # 5. Extract the corresponding values for the most frequent answer(s) from the pre-computed dictionaries
    most_frequent_uncertainty = {answer: average_uncertainty[answer] for answer in most_frequent_answers}
    most_frequent_self_consistency = {answer: self_consistency[answer] for answer in most_frequent_answers}
    
    if no_reason:
        most_frequent_reasoning_tokens = None
    else:
        most_frequent_reasoning_tokens = {answer: average_reasoning_tokens[answer] for answer in most_frequent_answers}

    # Correct answer?
    is_correct = row['answer_idx'] in most_frequent_answers

    # Strategy metrics
    strategy_counts = Counter(all_strategies) if all_strategies else Counter()
    unique_strategies = len(strategy_counts)
    total_strategy_mentions = len(all_strategies)
    
    # Return the computed metrics
    return {
        "is_correct": is_correct,
        "average_uncertainty": average_uncertainty,
        "self_consistency": self_consistency,
        "most_frequent_answers": most_frequent_answers,
        "most_frequent_uncertainty": most_frequent_uncertainty,
        "most_frequent_self_consistency": most_frequent_self_consistency,
        "most_frequent_reasoning_tokens": most_frequent_reasoning_tokens,
        "strategies": dict(strategy_counts),  # Add strategy counts
        "unique_strategies": unique_strategies,  # Number of unique strategies
        "total_strategy_mentions": total_strategy_mentions,  # Total strategy mentions
        "average_reasoning_tokens": average_reasoning_tokens,
        "individual_responses": individual_responses  # Individual response data
    }

In [ ]:
dfs_eval_false = []
medqa_scores = {}
overall_self_consistency = {}  # Store overall self-consistency

for df_eval in dfs_eval:
    # Get filename to use in output path (assumes 'source_file' column is set)
    filename = df_eval['source_file'].iloc[0]
    base_name = os.path.splitext(filename)[0]  # Remove '.json'
    
    # Build output path for metrics file
    eval_path = os.path.join(EVAL_PATH, f"metrics_{base_name}.jsonl")
    
    # Apply metrics calculation to each row
    df_eval['question_stats'] = df_eval.apply(lambda row: calculate_case_metrics(row), axis=1)
    
    # Write results to file
    with open(eval_path, "w") as eval_file:
        for _, row in df_eval.iterrows():  # removed asterisks
            result = {
                "question": row["question"],
                "id": row["id"],
                "answer_idx": row["answer_idx"],
                "meta_info": row["meta_info"],
                "options": row["options"],
                "response": row["response"],
                "question_stats": row["question_stats"]
            }
            eval_file.write(json.dumps(result) + "\n")
            
    # Filter for incorrect responses only
    df_false = df_eval[df_eval['question_stats'].apply(lambda x: not x['is_correct'])]
    dfs_eval_false.append(df_false)
    false_question_path = os.path.join(EVAL_PATH, f"false_questions_{base_name}.jsonl")
    with open(false_question_path, "w") as false_file:
        for _, row in df_false.iterrows():  # removed asterisks
            result = {
                "question": row["question"],
                "id": row["id"],
                "answer_idx": row["answer_idx"],
                "meta_info": row["meta_info"],
                "options": row["options"],
                "response": row["response"],
                "question_stats": row["question_stats"]
            }
            false_file.write(json.dumps(result) + "\n")
    
    # Calculate MedQA score
    correct_count = df_eval['question_stats'].apply(lambda x: x['is_correct']).sum()
    medqa_score = correct_count / len(df_eval)
    medqa_scores[base_name] = round(medqa_score * 100, 2)  # Store as percentage rounded to 2 decimals
    
    # Calculate overall self-consistency
    # Extract self-consistency values for most frequent answers from each question
    self_consistency_values = []
    for _, row in df_eval.iterrows():
        question_stats = row['question_stats']
        if question_stats and 'most_frequent_self_consistency' in question_stats:
            # Get self-consistency values for most frequent answers
            most_freq_self_consistency = question_stats['most_frequent_self_consistency']
            # Add all self-consistency values for most frequent answers
            for answer, consistency in most_freq_self_consistency.items():
                if consistency is not None:
                    self_consistency_values.append(consistency)
    
    # Calculate average self-consistency across all questions
    if self_consistency_values:
        avg_self_consistency = np.mean(self_consistency_values)
        overall_self_consistency[base_name] = round(avg_self_consistency, 4)
    else:
        overall_self_consistency[base_name] = None
    
    print(f"{base_name}: {correct_count}/{len(df_eval)} correct → MedQA score = {medqa_score:.2%}")
    if overall_self_consistency[base_name] is not None:
        print(f"  Overall self-consistency: {overall_self_consistency[base_name]:.4f}")
    else:
        print(f"  Overall self-consistency: No data available")

# Write MedQA scores to JSON file (now includes self-consistency)
scores_data = {
    "medqa_scores": medqa_scores,
    "overall_self_consistency": overall_self_consistency
}

scores_path = os.path.join(EVAL_PATH, "medqa_scores.json")
with open(scores_path, "w") as scores_file:
    json.dump(scores_data, scores_file, indent=2)

print(f"\nScores saved to: {scores_path}")

In [ ]:
# ============================================================================
# CREATE VISUALIZATIONS for medqa score and self-consistency 
# ============================================================================

# Prepare data for plotting
model_names = list(medqa_scores.keys())
medqa_score_values = [medqa_scores[name] for name in model_names]
self_consistency_values = [overall_self_consistency[name] for name in model_names]

# Filter out None values for self-consistency plotting
valid_self_consistency = [(name, value) for name, value in zip(model_names, self_consistency_values) 
                         if value is not None]

if len(model_names) > 0:
    
    # ============================================================================
    # PLOT 1: MedQA SCORES BAR CHART
    # ============================================================================
    
    plt.figure(figsize=(12, 8))
    
    # Use different colors for each bar (like SCT plots)
    colors = plt.cm.Set3(np.linspace(0, 1, len(model_names)))
    bars = plt.bar(range(len(model_names)), medqa_score_values, alpha=0.8, 
                   color=colors, edgecolor='black')
    plt.xlabel('Models', fontsize=12)
    plt.ylabel('MedQA Score (%)', fontsize=12)
    plt.title('MedQA Performance Comparison', fontsize=14, fontweight='bold')
    plt.xticks(range(len(model_names)), clean_labels, rotation=45, ha='right')
    plt.grid(True, alpha=0.3, axis='y')
    
    # Add value labels on bars
    for bar, score in zip(bars, medqa_score_values):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{score:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=11)
    
    plt.tight_layout()
    
    # Save the plot
    plots_dir = os.path.join(EVAL_PATH, 'plots')
    os.makedirs(plots_dir, exist_ok=True)
    plot_path_medqa = os.path.join(plots_dir, 'medqa_scores_comparison.png')
    plt.savefig(plot_path_medqa, dpi=300, bbox_inches='tight')
    print(f"MedQA scores plot saved to: {plot_path_medqa}")
    
    plt.show()
    
    # ============================================================================
    # PLOT 2: SELF-CONSISTENCY BAR CHART
    # ============================================================================
    
    if valid_self_consistency:
        valid_names = [name for name, value in valid_self_consistency]
        valid_values = [value for name, value in valid_self_consistency]
        
        plt.figure(figsize=(12, 8))
        
        # Use different colors for each bar
        colors = plt.cm.Set3(np.linspace(0, 1, len(valid_names)))
        bars2 = plt.bar(range(len(valid_names)), valid_values, alpha=0.8, 
                       color=colors, edgecolor='black')
        plt.xlabel('Models', fontsize=12)
        plt.ylabel('Overall Self-Consistency', fontsize=12)
        plt.title('Self-Consistency Comparison', fontsize=14, fontweight='bold')
        plt.xticks(range(len(valid_names)), clean_labels, rotation=45, ha='right')
        plt.grid(True, alpha=0.3, axis='y')
        plt.ylim(0, 1)  # Self-consistency is between 0 and 1
        
        # Add value labels on bars
        for bar, value in zip(bars2, valid_values):
            plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                    f'{value:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=11)
        
        plt.tight_layout()
        
        # Save the plot
        plot_path_consistency = os.path.join(plots_dir, 'medqa_self_consistency_comparison.png')
        plt.savefig(plot_path_consistency, dpi=300, bbox_inches='tight')
        print(f"Self-consistency plot saved to: {plot_path_consistency}")
        
        plt.show()
    
    else:
        print("No self-consistency data available for plotting")
    
    print(f"All MedQA plots saved to: {plots_dir}")

    # ============================================================================
    # PLOT 3: SELF-CONSISTENCY BOX PLOT (DISTRIBUTION)
    # ============================================================================
    
    if valid_self_consistency:
        # Create individual self-consistency distributions for box plot
        self_consistency_distributions = []
        valid_names_boxplot = []
        
        for model_name in model_names:
            if model_name in [name for name, _ in valid_self_consistency]:
                # Find the corresponding dataframe for this model
                model_df = next(df for df in dfs_eval if df['source_file'].iloc[0].startswith(model_name.split('.')[0]))
                individual_values = []
                
                for _, row in model_df.iterrows():
                    question_stats = row['question_stats']
                    if question_stats and 'most_frequent_self_consistency' in question_stats:
                        # Get self-consistency values for most frequent answers
                        most_freq_self_consistency = question_stats['most_frequent_self_consistency']
                        for answer, consistency in most_freq_self_consistency.items():
                            if consistency is not None:
                                individual_values.append(consistency)
                
                if individual_values:
                    self_consistency_distributions.append(individual_values)
                    valid_names_boxplot.append(model_name)
        
        if self_consistency_distributions:
            plt.figure(figsize=(12, 8))
            
            box_plot = plt.boxplot(self_consistency_distributions, tick_labels=clean_labels, 
                      patch_artist=True, showmeans=True)
            plt.xlabel('Models', fontsize=12)
            plt.ylabel('Self-Consistency', fontsize=12)
            plt.title('Self-Consistency Distribution (Most Frequent Answers)', fontsize=14, fontweight='bold')
            plt.xticks(rotation=45, ha='right')
            plt.grid(True, alpha=0.3, axis='y')
            plt.ylim(0, 1)  # Self-consistency is between 0 and 1
            
            # Color the boxplots
            colors = plt.cm.Set3(np.linspace(0, 1, len(valid_names_boxplot)))
            for patch, color in zip(box_plot['boxes'], colors):
                patch.set_facecolor(color)
                patch.set_alpha(0.7)
            
            # Add overall mean value labels (same values as bar chart)
            for i, name in enumerate(valid_names_boxplot):
                overall_mean = overall_self_consistency[name]
                plt.text(i+1, overall_mean, f'{overall_mean:.3f}', ha='center', va='center',
                        bbox=dict(boxstyle="round,pad=0.3", facecolor='white', alpha=0.8),
                        fontweight='bold', fontsize=10)
            
            plt.tight_layout()
            
            # Save the plot
            plot_path_boxplot = os.path.join(plots_dir, 'medqa_self_consistency_distribution.png')
            plt.savefig(plot_path_boxplot, dpi=300, bbox_inches='tight')
            print(f"Self-consistency distribution plot saved to: {plot_path_boxplot}")
            
            plt.show()
        else:
            print("No individual self-consistency data available for box plot")
    else:
        print("No self-consistency data available for box plot")
        
else:
    print("No MedQA data available for plotting")

print(f"\nProcessed {len(model_names)} MedQA models successfully")

In [ ]:
def load_all_metrics_jsonl(folder_path):
    """Load all metrics files without complex name parsing"""
    all_data = {}
    
    print(f"Looking for metrics files in: {folder_path}")
    
    for fname in os.listdir(folder_path):
        if fname.startswith("metrics_") and fname.endswith(".jsonl"):
            print(f"Found metrics file: {fname}")
            
            # Extract a simple model identifier
            # Remove "metrics_" prefix and ".jsonl" suffix
            base_name = fname.replace("metrics_", "").replace(".jsonl", "")
            
            # Determine model type based on filename content
            if "reason" in fname.lower():
                model_key = "reasoning_model"
            elif "base" in fname.lower():
                model_key = "base_model"
            else:
                model_key = base_name  # fallback
            
            try:
                with open(os.path.join(folder_path, fname), 'r') as f:
                    data = [json.loads(line) for line in f]
                    all_data[model_key] = data
                    print(f"✅ Loaded {len(data)} records for: {model_key}")
            except Exception as e:
                print(f"❌ Error loading {fname}: {e}")
    
    return all_data

In [ ]:
all_model_data = load_all_metrics_jsonl(EVAL_PATH)


In [ ]:
def compute_calibration_metrics(y_true, y_conf, n_bins=8):
    """
    Compute ECE, OE, and Brier score using uniform bins.

    Parameters
    ----------
    y_true  : array-like, binary ground-truth (0/1)
    y_conf  : array-like, predicted confidence in [0, 1]
    n_bins  : number of equal-width bins (default 8)

    Returns
    -------
    dict with keys: ECE, OE, Brier
    """
    if len(y_true) != len(y_conf):
        raise ValueError("y_true and y_conf must have same length")

    y_true = np.asarray(y_true)
    y_conf = np.asarray(y_conf)
    y_conf = np.clip(y_conf, 0, 1)

    bins    = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(y_conf, bins) - 1
    bin_ids = np.clip(bin_ids, 0, n_bins - 1)  # handle y_conf == 1.0

    ece, oe, total = 0.0, 0.0, len(y_true)
    for i in range(n_bins):
        mask     = bin_ids == i
        bin_size = int(np.sum(mask))
        if bin_size == 0:
            continue
        avg_conf = float(np.mean(y_conf[mask]))
        avg_acc  = float(np.mean(y_true[mask]))
        ece += (bin_size / total) * np.abs(avg_conf - avg_acc)
        if avg_conf > avg_acc:
            oe += (bin_size / total) * (avg_conf - avg_acc)

    brier = float(np.mean((y_conf - y_true) ** 2))
    return {'ECE': ece, 'OE': oe, 'Brier': brier}

In [ ]:

from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.calibration import calibration_curve
import random

# Check data content
for model_name, data in all_model_data.items():
    print(f"Model '{model_name}': {len(data)} records")

def flatten_per_question(data):
    """Original method using majority answer"""
    records = []
    for case in data:
        stats = case['question_stats']
        majority_answer = stats['most_frequent_answers'][0]
        avg_uncertainty = stats['average_uncertainty'][majority_answer] / 100.0
        self_consistency = stats['self_consistency'][majority_answer]
        records.append({
            'question_id': case['id'],
            'predicted_answer': majority_answer,
            'confidence': avg_uncertainty,
            'self_consistency': self_consistency,
            'is_correct': stats['is_correct']
        })
    return pd.DataFrame(records)

def flatten_per_question_random(data, random_seed=42):
    """Random response method - selects one random response per question"""
    random.seed(random_seed)  # For reproducibility
    records = []
    
    for case in data:
        stats = case['question_stats']
        
        # Get individual responses from the new field we added
        if 'individual_responses' in stats and stats['individual_responses']:
            individual_responses = stats['individual_responses']
            
            # Randomly select one response key
            response_keys = list(individual_responses.keys())
            selected_key = random.choice(response_keys)
            selected_response = individual_responses[selected_key]
            
            # Get the selected response data
            answer = selected_response['answer']
            uncertainty = selected_response['uncertainty']
            
            # Convert uncertainty to confidence (0-1 scale)
            if uncertainty is not None:
                confidence = uncertainty / 100.0
            else:
                confidence = 0.5  # Default neutral confidence for missing uncertainty
            
            # Check if this random response was correct
            is_correct = (answer == case['answer_idx'])
            
            # Get self-consistency for this specific answer (how often this answer was chosen)
            self_consistency = stats['self_consistency'].get(answer, 0)
            
            records.append({
                'question_id': case['id'],
                'predicted_answer': answer,
                'confidence': confidence,
                'self_consistency': self_consistency,
                'is_correct': is_correct,
                'selected_response': selected_key
            })
        else:
            # Fallback to majority method if individual_responses not available
            majority_answer = stats['most_frequent_answers'][0]
            avg_uncertainty = stats['average_uncertainty'][majority_answer] / 100.0
            records.append({
                'question_id': case['id'],
                'predicted_answer': majority_answer,
                'confidence': avg_uncertainty,
                'self_consistency': stats['self_consistency'][majority_answer],
                'is_correct': stats['is_correct'],
                'selected_response': 'majority_fallback'
            })
    
    return pd.DataFrame(records)

def flatten_all_responses(data):
    """Uses ALL individual responses - treats each response as independent sample"""
    records = []
    
    for case in data:
        stats = case['question_stats']
        
        # Get individual responses from the new field we added
        if 'individual_responses' in stats and stats['individual_responses']:
            individual_responses = stats['individual_responses']
            
            # Include ALL responses, not just one random selection
            for response_key, response_data in individual_responses.items():
                answer = response_data['answer']
                uncertainty = response_data['uncertainty']
                
                # Convert uncertainty to confidence (0-1 scale)
                if uncertainty is not None:
                    confidence = uncertainty / 100.0
                else:
                    confidence = 0.5  # Default neutral confidence for missing uncertainty
                
                # Check if this response was correct
                is_correct = (answer == case['answer_idx'])
                
                # Get self-consistency for this specific answer
                self_consistency = stats['self_consistency'].get(answer, 0)
                
                records.append({
                    'question_id': case['id'],
                    'predicted_answer': answer,
                    'confidence': confidence,
                    'self_consistency': self_consistency,
                    'is_correct': is_correct,
                    'response_key': response_key
                })
        else:
            # Fallback to majority method if individual_responses not available
            majority_answer = stats['most_frequent_answers'][0]
            avg_uncertainty = stats['average_uncertainty'][majority_answer] / 100.0
            records.append({
                'question_id': case['id'],
                'predicted_answer': majority_answer,
                'confidence': avg_uncertainty,
                'self_consistency': stats['self_consistency'][majority_answer],
                'is_correct': stats['is_correct'],
                'response_key': 'majority_fallback'
            })
    
    return pd.DataFrame(records)

def plot_case_level_comparison(model_data_dict, n_bins=8, signal='confidence',
                               method='majority', strategy='uniform'):
    """
    Create separate calibration and ROC plots.

    Parameters
    ----------
    n_bins    : number of bins for ECE calculation and calibration curve plot
    signal    : 'confidence' or 'self_consistency'
    method    : 'majority', 'random', or 'all_responses'
    strategy  : 'uniform' (equal-width) or 'quantile' (equal-count),
                passed to sklearn calibration_curve for the plot.
                ECE is always computed with uniform bins.
    """

    model_names = list(model_data_dict.keys())

    try:
        plot_labels = clean_labels[:len(model_names)] if len(clean_labels) >= len(model_names) else model_names
    except NameError:
        print("Warning: clean_labels not found, using original model names")
        plot_labels = model_names

    label_mapping = dict(zip(model_names, plot_labels))

    if method == 'random':
        flatten_func = flatten_per_question_random
        method_label = "Random Response"
    elif method == 'all_responses':
        flatten_func = flatten_all_responses
        method_label = "All Individual Responses"
    else:
        flatten_func = flatten_per_question
        method_label = "Majority Answer"

    # ---- Plot 1: Calibration Curve ----
    plt.figure(figsize=(12, 8))

    for model_name, data in model_data_dict.items():
        df     = flatten_func(data)
        y_true = df['is_correct'].astype(int).values
        y_conf = df[signal].values

        metrics = compute_calibration_metrics(y_true, y_conf, n_bins=n_bins)
        clean_label = label_mapping[model_name]

        prob_true, prob_pred = calibration_curve(
            y_true, y_conf, n_bins=n_bins, strategy=strategy)

        # hollow circles, size encodes bin occupancy
        bin_sizes = []
        for pp in prob_pred:
            half = (prob_pred[1] - prob_pred[0]) / 2 if len(prob_pred) > 1 else 0.5
            bin_sizes.append(int(np.sum(np.abs(y_conf - pp) <= half)))
        max_bs = max(bin_sizes) if bin_sizes else 1
        sizes = [30 + 270 * (bs / max_bs) for bs in bin_sizes]

        color = plt.gca()._get_lines.get_next_color()
        plt.plot(prob_pred, prob_true, color=color, linewidth=1.5,
                 label=f"{clean_label} (ECE={metrics['ECE']:.3f})")
        plt.scatter(prob_pred, prob_true, s=sizes, facecolors='none',
                    edgecolors=color, linewidths=1.5, zorder=5)

    plt.plot([0, 1], [0, 1], '--', color='gray', alpha=0.8, linewidth=1.5,
             label='Perfect calibration')
    plt.xlabel("Confidence Score", fontsize=12)
    plt.ylabel("Accuracy", fontsize=12)
    strat_label = "quantile bins" if strategy == 'quantile' else "uniform bins"
    #          fontsize=13, fontweight='bold')
    plt.legend(fontsize=11)
    plt.grid(True, alpha=0.3)

    output_dir = os.path.join(EVAL_PATH, 'plots')
    os.makedirs(output_dir, exist_ok=True)
    plt.savefig(os.path.join(output_dir, f'calibration_curve_{signal}_{method}.png'),
                dpi=300, bbox_inches='tight')
    plt.show()

    # ---- Plot 2: ROC Curve ----
    plt.figure(figsize=(12, 8))

    for model_name, data in model_data_dict.items():
        df     = flatten_func(data)
        y_true = df['is_correct'].astype(int).values
        y_conf = df[signal].values

        clean_label = label_mapping[model_name]
        fpr, tpr, _ = roc_curve(y_true, y_conf)
        auc = roc_auc_score(y_true, y_conf)
        plt.plot(fpr, tpr, label=f'{clean_label} (AUROC={auc:.3f})', linewidth=2)

    plt.plot([0, 1], [0, 1], '--', color='gray', alpha=0.8)
    plt.xlabel("FPR", fontsize=12)
    plt.ylabel("TPR", fontsize=12)
    #          fontsize=14, fontweight='bold')
    plt.legend(fontsize=11)
    plt.grid(True, alpha=0.3)

    plt.savefig(os.path.join(output_dir, f'roc_curve_{signal}_{method}.png'),
                dpi=300, bbox_inches='tight')
    plt.show()

    print(f"Plots saved to: {output_dir}")
    print(f"  - calibration_curve_{signal}_{method}.png")
    print(f"  - roc_curve_{signal}_{method}.png")

In [ ]:
# Parameters: n_bins, signal, method, strategy ('uniform' or 'quantile')
plot_case_level_comparison(all_model_data, n_bins=8, signal='self_consistency', method="majority", strategy='uniform')
plot_case_level_comparison(all_model_data, n_bins=8, signal='self_consistency', method="all_responses", strategy='uniform')
plot_case_level_comparison(all_model_data, n_bins=8, signal='confidence', method="majority", strategy='uniform')
plot_case_level_comparison(all_model_data, n_bins=8, signal='confidence', method="all_responses", strategy='uniform')

In [ ]:
import os
import pandas as pd
import json
import glob
import re
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import random
import tempfile

def medqa_score(file_path, N=100):
    """
    Complete MedQA analysis using the actual process_response function for extraction.
    Compares majority vote vs random sampling.
    
    Args:
        file_path: Path to directory containing response files
        N: Number of random samples for individual response analysis
    """
    
    # ============================================================================
    # 1. LOAD RESPONSE FILES
    # ============================================================================
    print(f"Loading response files from: {file_path}")
    response_files = glob.glob(os.path.join(file_path, 'responses_*.json'))
    
    if not response_files:
        print(f"No response files found in {file_path}")
        return
    
    print(f"Found {len(response_files)} response files")
    
    all_results = {}
    
    for filepath in response_files:
        filename = os.path.basename(filepath)
        display_name = os.path.splitext(filename)[0].replace("responses_", "")
        
        print(f"\nProcessing: {filename}")
        
        # Load file
        with open(filepath, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        print(f"  Loaded {len(data)} questions")
        
        # ============================================================================
        # 2. CONVERT TO DATAFRAME FORMAT AND CALL process_response
        # ============================================================================
        
        # Convert to DataFrame format that process_response expects
        df_data = []
        for item in data:
            df_data.append({
                'question': item.get('question'),
                'id': item.get('id'),
                'answer_idx': item.get('answer_idx'),
                'meta_info': item.get('meta_info'),
                'options': item.get('options'),
                'response': item.get('response')
            })
        
        df = pd.DataFrame(df_data)
        
        print(f"  Calling process_response() for {len(df)} questions...")
        
        # Create a temporary error file
        with tempfile.NamedTemporaryFile(mode='w', delete=False, suffix='.jsonl') as temp_error_file:
            temp_error_path = temp_error_file.name
        
        # Apply process_response to each row
        processed_results = []
        with open(temp_error_path, 'w') as error_file:
            for _, row in df.iterrows():
                result = process_response(row, error_file)
                if result is not None:
                    processed_results.append(result)
        
        # Clean up temp file
        os.unlink(temp_error_path)
        
        print(f"  Successfully processed: {len(processed_results)}/{len(df)} questions")
        
        # ============================================================================
        # 3. EXTRACT ANSWERS FROM process_response RESULTS
        # ============================================================================
        
        processed_questions = []
        
        for result in processed_results:
            question_data = {
                'id': result['id'],
                'question': result['question'],
                'answer_idx': result['answer_idx'],
                'responses': {}  # Will store extracted answers for each response
            }
            
            # Extract answers from process_response results
            results_dict = result.get('results', {})
            for resp_key, resp_data in results_dict.items():
                if resp_data.get('answer'):  # Only include responses where answer was extracted
                    question_data['responses'][resp_key] = {
                        'answer': resp_data['answer'],
                        'is_correct': resp_data.get('is_correct', False)
                    }
            
            # Only include questions where we extracted at least some answers
            if question_data['responses']:
                processed_questions.append(question_data)
        
        print(f"  Questions with extracted answers: {len(processed_questions)}")
        
        # ============================================================================
        # 4. CALCULATE MAJORITY VOTE SCORE (SAME AS MAIN SCRIPT)
        # ============================================================================
        
        majority_results = []
        
        for question in processed_questions:
            # Count answers
            answer_counts = Counter()
            for resp_data in question['responses'].values():
                answer_counts[resp_data['answer']] += 1
            
            if answer_counts:
                # Find most frequent answer(s) - same logic as calculate_case_metrics
                most_frequent_answer_count = answer_counts.most_common(1)[0][1]
                most_frequent_answers = [answer for answer, count in answer_counts.items() 
                                       if count == most_frequent_answer_count]
                
                # Check if correct answer is in most frequent (same as calculate_case_metrics)
                is_correct = question['answer_idx'] in most_frequent_answers
                
                majority_results.append({
                    'question_id': question['id'],
                    'most_frequent_answers': most_frequent_answers,
                    'is_correct': is_correct,
                    'total_responses': len(question['responses'])
                })
        
        # Calculate majority vote score
        correct_count = sum(1 for result in majority_results if result['is_correct'])
        majority_score = correct_count / len(majority_results) * 100  # Convert to percentage
        
        print(f"  Majority vote accuracy: {majority_score:.2f}% ({correct_count}/{len(majority_results)})")
        
        # ============================================================================
        # 5. CALCULATE RANDOM SAMPLING SCORES
        # ============================================================================
        
        print(f"  Calculating {N} random samples...")
        random_scores = []
        
        for sample_num in range(N):
            sample_correct = 0
            sample_total = 0
            
            for question in processed_questions:
                if question['responses']:
                    # Randomly pick one response
                    random_response_key = random.choice(list(question['responses'].keys()))
                    random_response = question['responses'][random_response_key]
                    
                    if random_response['is_correct']:
                        sample_correct += 1
                    sample_total += 1
            
            if sample_total > 0:
                sample_accuracy = sample_correct / sample_total * 100
                random_scores.append(sample_accuracy)
        
        random_mean = np.mean(random_scores)
        random_std = np.std(random_scores)
        
        print(f"  Random sampling: {random_mean:.2f}% ± {random_std:.2f}% (mean ± std of {N} samples)")
        
        # Store results
        all_results[display_name] = {
            'majority_score': majority_score,
            'random_scores': random_scores,
            'random_mean': random_mean,
            'random_std': random_std,
            'n_questions': len(majority_results),
            'n_samples': N
        }
    
    # ============================================================================
    # 6. CREATE VISUALIZATIONS - SEPARATE PLOTS
    # ============================================================================
    
    if not all_results:
        print("No results to plot")
        return
    
    # Prepare data for plotting
    file_names = list(all_results.keys())
    majority_scores = [all_results[name]['majority_score'] for name in file_names]
    clean_labels = get_clean_labels(file_names) #for clean labels according to file

    # Create plots directory
    plots_dir = os.path.join(EVAL_PATH, 'plots')
    os.makedirs(plots_dir, exist_ok=True)
    
    # ---- Plot 1: Bar chart for majority vote scores (separate plot) ----
    plt.figure(figsize=(12, 8))
    
    # Use different colors for each bar
    colors = plt.cm.Set3(np.linspace(0, 1, len(file_names)))
    bars = plt.bar(range(len(file_names)), majority_scores, alpha=0.8, 
                   color=colors, edgecolor='black')
    plt.xlabel('Response Files', fontsize=12)
    plt.ylabel('Accuracy (%)', fontsize=12)
    plt.title('Majority Vote Accuracy Comparison', fontsize=14, fontweight='bold')
    plt.xticks(range(len(file_names)), clean_labels, rotation=45, ha='right')
    plt.grid(True, alpha=0.3, axis='y')
    
    # Add value labels on bars
    for bar, score in zip(bars, majority_scores):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{score:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=11)
    
    plt.tight_layout()
    # Save the plot
    bar_chart_path = os.path.join(plots_dir, 'medqa_majority_vote_comparison.png')
    plt.savefig(bar_chart_path, dpi=300, bbox_inches='tight')
    print(f"Bar chart saved to: {bar_chart_path}")

    plt.show()
    
    # ---- Plot 2: Boxplot for random sampling scores (separate plot) ----
    plt.figure(figsize=(12, 8))
    
    random_data = [all_results[name]['random_scores'] for name in file_names]
    
    box_plot = plt.boxplot(random_data, tick_labels=clean_labels, patch_artist=True)
    plt.xlabel('Response Files', fontsize=12)
    plt.ylabel('Accuracy (%)', fontsize=12)
    plt.title(f'Random Sampling Accuracy Distribution\n({N} samples per file)', 
              fontsize=14, fontweight='bold')
    plt.xticks(rotation=45, ha='right')
    plt.grid(True, alpha=0.3, axis='y')
    
    # Color the boxplots with same colors as bar chart
    colors = plt.cm.Set3(np.linspace(0, 1, len(file_names)))
    for patch, color in zip(box_plot['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    
    # Add mean value labels
    for i, name in enumerate(file_names):
        mean_val = all_results[name]['random_mean']
        plt.text(i+1, mean_val, f'{mean_val:.1f}%', ha='center', va='center',
                bbox=dict(boxstyle="round,pad=0.3", facecolor='white', alpha=0.8),
                fontweight='bold', fontsize=9)
    
    plt.tight_layout()
    # Save the plot
    boxplot_path = os.path.join(plots_dir, 'medqa_random_sampling_distribution.png')
    plt.savefig(boxplot_path, dpi=300, bbox_inches='tight')
    print(f"Box plot saved to: {boxplot_path}")

    plt.show()
    
    # ============================================================================
    # SUMMARY TABLE
    # ============================================================================
    
    print("\n" + "="*80)
    print("SUMMARY RESULTS")
    print("="*80)
    print(f"{'File Name':<40} {'Majority':<12} {'Random Mean':<12} {'Random Std':<12} {'Questions':<10}")
    print("-" * 80)
    
    for name in file_names:
        result = all_results[name]
        print(f"{name:<40} {result['majority_score']:8.2f}% {result['random_mean']:9.2f}% "
              f"{result['random_std']:9.2f}% {result['n_questions']:8d}")
    
    print("="*80)

    print(f"\n📁 Plots saved to: {plots_dir}")
    print(f"  - medqa_majority_vote_comparison.png")
    print(f"  - medqa_random_sampling_distribution.png")
    
    return all_results


def check_process_response_available():
    """Check if process_response function is available"""
    try:
        process_response
        print("✅ process_response function is available")
        return True
    except NameError:
        print("❌ ERROR: process_response function not found!")
        print("Please import it first: from your_main_script import process_response")
        return False

In [ ]:
# Run analysis on response files


response_file_path = "./data/output/gpt4-1_comparison"  # Adjust as needed

# Run the complete analysis
results = medqa_score(response_file_path, N=100)

In [ ]:

def normalize_strategy_to_official(strategy):
    """
    Normalize strategy names to map to the 12 official clinical reasoning categories.
    Returns 'other' for strategies that don't clearly fit the official categories.
    """
    if not strategy or not isinstance(strategy, str):
        return "other"
    
    # Convert to lowercase and clean
    normalized = strategy.lower().strip()
    normalized = re.sub(r'[^\w\s-]', '', normalized)  # Remove punctuation except hyphens
    normalized = re.sub(r'\s+', ' ', normalized)  # Normalize whitespace
    normalized = normalized.strip()
    
    # Official strategy categories (12 total: 10 + 2 new)
    official_categories = {
        'deductive': 'Deductive Reasoning',
        'hypothetico-deductive': 'Hypothetico-Deductive Reasoning', 
        'inductive': 'Inductive Reasoning',
        'abductive': 'Abductive Reasoning',
        'probabilistic': 'Probabilistic Reasoning',
        'rule-based': 'Rule-Based / Categorical / Deterministic Reasoning',
        'causal': 'Causal Reasoning',
        'heuristic': 'Heuristic / Pattern Recognition (Fast Thinking)',
        'red-flag': 'Red Flag / Rule-Out Reasoning',
        'guideline-based': 'Guideline-Based Reasoning',
        'ethical': 'Ethical Reasoning',
        'patient-centered': 'Patient-Centered Risk-Benefit Analysis'
    }
    
    # Comprehensive mapping to official categories
    strategy_mapping = {
        # Deductive Reasoning
        'deductive': 'deductive',
        'deduction': 'deductive',
        'deductive reasoning': 'deductive',
        'top down': 'deductive',
        'top-down': 'deductive',
        'rule application': 'deductive',
        'principle based': 'deductive',
        'principle-based': 'deductive',
        
        # Hypothetico-Deductive Reasoning
        'hypothetico-deductive': 'hypothetico-deductive',
        'hypothetico deductive': 'hypothetico-deductive',
        'hypothetical-deductive': 'hypothetico-deductive',
        'hypothetical deductive': 'hypothetico-deductive',
        'hypothesis testing': 'hypothetico-deductive',
        'hypothesis-driven': 'hypothetico-deductive',
        'hypothesis driven': 'hypothetico-deductive',
        'differential reasoning': 'hypothetico-deductive',
        'multiple hypothesis': 'hypothetico-deductive',
        'hypothetico-deductive reasoning': 'hypothetico-deductive',
        'differential diagnosis': 'hypothetico-deductive',
        'differential': 'hypothetico-deductive',
        
        # Inductive Reasoning
        'inductive': 'inductive',
        'induction': 'inductive',
        'inductive reasoning': 'inductive',
        'bottom up': 'inductive',
        'bottom-up': 'inductive',
        'data driven': 'inductive',
        'data-driven': 'inductive',
        'pattern synthesis': 'inductive',
        
        # Abductive Reasoning
        'abductive': 'abductive',
        'abduction': 'abductive',
        'abductive reasoning': 'abductive',
        'inference to best explanation': 'abductive',
        'best explanation': 'abductive',
        'backwards reasoning': 'abductive',
        'reverse reasoning': 'abductive',
        'explanatory reasoning': 'abductive',
        
        # Probabilistic Reasoning
        'probabilistic': 'probabilistic',
        'probability': 'probabilistic',
        'probabilistic reasoning': 'probabilistic',
        'bayesian': 'probabilistic',
        'likelihood': 'probabilistic',
        'statistical': 'probabilistic',
        'risk based': 'probabilistic',
        'risk-based': 'probabilistic',
        'pretest probability': 'probabilistic',
        'post-test probability': 'probabilistic',
        'prevalence': 'probabilistic',
        
        # Rule-Based / Categorical / Deterministic Reasoning
        'rule-based': 'rule-based',
        'rule based': 'rule-based',
        'categorical': 'rule-based',
        'deterministic': 'rule-based',
        'criteria based': 'rule-based',
        'criteria-based': 'rule-based',
        'threshold': 'rule-based',
        'cutoff': 'rule-based',
        'scoring system': 'rule-based',
        'decision rule': 'rule-based',
        'clinical decision rule': 'rule-based',
        'algorithmic': 'rule-based',
        'algorithm': 'rule-based',
        
        # Causal Reasoning (expanded to include anatomical/physiological knowledge)
        'causal': 'causal',
        'causation': 'causal',
        'causal reasoning': 'causal',
        'pathophysiologic': 'causal',
        'pathophysiology': 'causal',
        'pathophysiological': 'causal',
        'mechanistic': 'causal',
        'mechanism': 'causal',
        'cause effect': 'causal',
        'cause-effect': 'causal',
        'physiologic': 'causal',
        'physiological': 'causal',
        'anatomical': 'causal',
        'anatomy': 'causal',
        'anatomical knowledge': 'causal',
        'physiological knowledge': 'causal',
        'pathophysiological knowledge': 'causal',
        'biochemical': 'causal',
        'molecular': 'causal',
        'pharmacological': 'causal',
        'pharmacology': 'causal',
        'pharmacokinetic': 'causal',
        'pharmacodynamic': 'causal',
        
        # Heuristic / Pattern Recognition (Fast Thinking)
        'heuristic': 'heuristic',
        'heuristics': 'heuristic',
        'pattern recognition': 'heuristic',
        'pattern matching': 'heuristic',
        'pattern': 'heuristic',
        'recognition': 'heuristic',
        'intuitive': 'heuristic',
        'intuition': 'heuristic',
        'fast thinking': 'heuristic',
        'system 1': 'heuristic',
        'gut feeling': 'heuristic',
        'clinical intuition': 'heuristic',
        'gestalt': 'heuristic',
        
        # Red Flag / Rule-Out Reasoning
        'red flag': 'red-flag',
        'red-flag': 'red-flag',
        'rule out': 'red-flag',
        'rule-out': 'red-flag',
        'ruling out': 'red-flag',
        'exclusion': 'red-flag',
        'exclude': 'red-flag',
        'worst case': 'red-flag',
        'worst-case': 'red-flag',
        'emergency': 'red-flag',
        'critical': 'red-flag',
        'life threatening': 'red-flag',
        'life-threatening': 'red-flag',
        'safety first': 'red-flag',
        'dont miss': 'red-flag',
        'cannot miss': 'red-flag',
        
        # Guideline-Based Reasoning
        'guideline-based': 'guideline-based',
        'guideline based': 'guideline-based',
        'guideline': 'guideline-based',
        'guidelines': 'guideline-based',
        'protocol': 'guideline-based',
        'protocols': 'guideline-based',
        'evidence-based': 'guideline-based',
        'evidence based': 'guideline-based',
        'standard of care': 'guideline-based',
        'best practice': 'guideline-based',
        'best practices': 'guideline-based',
        'recommendation': 'guideline-based',
        'recommendations': 'guideline-based',
        'consensus': 'guideline-based',
        
        # NEW: Ethical Reasoning
        'ethical': 'ethical',
        'ethics': 'ethical',
        'ethical reasoning': 'ethical',
        'medical ethics': 'ethical',
        'bioethics': 'ethical',
        'ethical principles': 'ethical',
        'autonomy': 'ethical',
        'beneficence': 'ethical',
        'non-maleficence': 'ethical',
        'nonmaleficence': 'ethical',
        'justice': 'ethical',
        'veracity': 'ethical',
        'informed consent': 'ethical',
        'patient rights': 'ethical',
        'professional integrity': 'ethical',
        'moral': 'ethical',
        'morality': 'ethical',
        
        # NEW: Patient-Centered Risk-Benefit Analysis
        'risk': 'patient-centered',
        'benefit': 'patient-centered',
        'risk benefit': 'patient-centered',
        'risk-benefit': 'patient-centered',
        'patient centered': 'patient-centered',
        'patient-centered': 'patient-centered',
        'patient benefit': 'patient-centered',
        'patient risk': 'patient-centered',
        'harm': 'patient-centered',
        'adverse effects': 'patient-centered',
        'side effects': 'patient-centered',
        'complications': 'patient-centered',
        'patient safety': 'patient-centered',
        'quality of life': 'patient-centered',
        'patient preferences': 'patient-centered',
        'shared decision': 'patient-centered',
        'individualized': 'patient-centered',
        'personalized': 'patient-centered',
    }
    
    # Try exact match first
    if normalized in strategy_mapping:
        category = strategy_mapping[normalized]
        return official_categories[category]
    
    # Try partial matches for compound strategies - LONGEST FIRST to avoid conflicts
    # Sort by length descending so "hypothetico-deductive" is checked before "deductive"
    sorted_keys = sorted(strategy_mapping.items(), key=lambda x: len(x[0]), reverse=True)
    for key, category in sorted_keys:
        if key in normalized:
            return official_categories[category]
    
    # Check if it contains key words from official categories (also longest first)
    for key, category in sorted_keys:
        if any(word in normalized for word in key.split()):
            return official_categories[category]
    
    # If no mapping found, return 'other'
    return "other"

def analyze_strategies_official_categories_medqa(model_data_dict, output_dir):
    """
    Analyze reasoning strategies using official clinical reasoning categories.
    Creates 4 separate plots instead of subplots, using clean labels.
    """
    
    # Official clinical reasoning categories - UPDATED to include 12 categories
    official_categories = [
        'Deductive Reasoning',
        'Hypothetico-Deductive Reasoning', 
        'Inductive Reasoning',
        'Abductive Reasoning',
        'Probabilistic Reasoning',
        'Rule-Based / Categorical / Deterministic Reasoning',
        'Causal Reasoning',
        'Heuristic / Pattern Recognition (Fast Thinking)',
        'Red Flag / Rule-Out Reasoning',
        'Guideline-Based Reasoning',
        'Ethical Reasoning',
        'Risk-Benefit Analysis'
    ]
    
    short_labels = {
        'Deductive Reasoning': 'Deductive',
        'Hypothetico-Deductive Reasoning': 'Hypothetico-\nDeductive', 
        'Inductive Reasoning': 'Inductive',
        'Abductive Reasoning': 'Abductive',
        'Probabilistic Reasoning': 'Probabilistic',
        'Rule-Based / Categorical / Deterministic Reasoning': 'Rule-Based /\nCategorical',
        'Causal Reasoning': 'Causal',
        'Heuristic / Pattern Recognition (Fast Thinking)': 'Heuristic /\nPattern Recog',
        'Red Flag / Rule-Out Reasoning': 'Red Flag /\nRule-Out',
        'Guideline-Based Reasoning': 'Guideline-\nBased',
        'Ethical Reasoning': 'Ethical',
        'Risk-Benefit Analysis': 'Risk-Benefit\nAnalysis',
        'other': 'Other'
    }
    
    # Get model names and create clean labels for this function
    model_names = list(model_data_dict.keys())
    
    # Use clean_labels if available, otherwise fall back to model names
    try:
        plot_labels = clean_labels[:len(model_names)] if len(clean_labels) >= len(model_names) else model_names
    except NameError:
        print("Warning: clean_labels not found, using original model names")
        plot_labels = model_names
    
    # Create label mapping
    label_mapping = dict(zip(model_names, plot_labels))
    
    # First pass: Check which models actually have strategies
    models_with_strategies = []
    for model_name, data in model_data_dict.items():
        strategy_count = 0
        for case in data:
            if case.get('question_stats'):
                qs = case['question_stats']
                
                # Check for strategies in the actual format they're stored
                if qs.get('strategies') and isinstance(qs['strategies'], dict):
                    # Sum up all strategy counts
                    strategy_count += sum(qs['strategies'].values())
                elif qs.get('most_frequent_strategies') and qs['most_frequent_strategies']:
                    # Original format (if it exists)
                    strategy_count += len(qs['most_frequent_strategies'])
        
        if strategy_count > 0:
            models_with_strategies.append(model_name)
            print(f"✅ {label_mapping[model_name]}: Found {strategy_count} strategies")
        else:
            print(f"⏭️ {label_mapping[model_name]}: No strategies found - skipping analysis")
    
    if not models_with_strategies:
        print("\n❌ No models with strategies found. Strategy analysis cannot be performed.")
        return
    
    print(f"\n📊 Analyzing strategies for {len(models_with_strategies)} model(s) with strategy data...")
    
    # Second pass: Analyze only models with strategies
    for model_name in models_with_strategies:
        data = model_data_dict[model_name]
        clean_model_name = label_mapping[model_name]
        
        print(f"\n🔍 Analyzing reasoning strategies for {clean_model_name}...")
        
        # Extract strategies from each case
        all_strategies = []
        strategy_correctness = []
        
        for case in data:
            if case.get('question_stats'):
                qs = case['question_stats']
                is_correct = qs.get('is_correct', False)
                
                # Handle the actual data format: strategies as dict with counts
                if qs.get('strategies') and isinstance(qs['strategies'], dict):
                    for strategy_name, count in qs['strategies'].items():
                        # Add each strategy the number of times it was mentioned
                        for _ in range(count):
                            normalized_strategy = normalize_strategy_to_official(strategy_name)
                            all_strategies.append(normalized_strategy)
                            strategy_correctness.append((normalized_strategy, is_correct))
                
                # Also handle original format if it exists
                elif qs.get('most_frequent_strategies') and qs['most_frequent_strategies']:
                    for strategy in qs['most_frequent_strategies']:
                        normalized_strategy = normalize_strategy_to_official(strategy)
                        all_strategies.append(normalized_strategy)
                        strategy_correctness.append((normalized_strategy, is_correct))
        
        # Count strategies
        strategy_counts_all = Counter(all_strategies)
        strategy_counts_correct = Counter([s for s, correct in strategy_correctness if correct])
        strategy_counts_incorrect = Counter([s for s, correct in strategy_correctness if not correct])
        
        # Only include categories that are actually used (from our 12 official categories + other)
        used_categories = [cat for cat in official_categories + ['other'] if strategy_counts_all[cat] > 0]
        
        # Use clean label for display
        clean_model_name = label_mapping[model_name]
        
        # Plot 1: Overall distribution (separate plot)
        plt.figure(figsize=(12, 8))
        counts = [strategy_counts_all[cat] for cat in used_categories]
        labels = [short_labels[cat] for cat in used_categories]
        
        bars1 = plt.bar(range(len(used_categories)), counts, alpha=0.8, 
                       color=plt.cm.Set3(np.linspace(0, 1, len(used_categories))))
        plt.title(f'Distribution of Reasoning Strategies - {clean_model_name}', 
                 fontsize=14, fontweight='bold')
        plt.xlabel('Strategy Categories', fontsize=12)
        plt.ylabel('Frequency', fontsize=12)
        plt.xticks(range(len(used_categories)), labels, rotation=45, ha='right')
        plt.grid(True, alpha=0.3, axis='y')
        
        # Add count labels
        for bar, count in zip(bars1, counts):
            plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(counts)*0.01,
                    str(count), ha='center', va='bottom', fontsize=10, fontweight='bold')
        
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, f'strategy_distribution_{model_name}.png'), 
                    dpi=300, bbox_inches='tight')
        plt.show()
        
        # Plot 2: Correct vs Incorrect (separate plot)
        plt.figure(figsize=(12, 8))
        correct_counts = [strategy_counts_correct.get(cat, 0) for cat in used_categories]
        incorrect_counts = [strategy_counts_incorrect.get(cat, 0) for cat in used_categories]
        
        x = np.arange(len(used_categories))
        width = 0.35
        
        bars2a = plt.bar(x - width/2, correct_counts, width, label='Correct Answer', 
                        alpha=0.8, color='lightgreen')
        bars2b = plt.bar(x + width/2, incorrect_counts, width, label='Incorrect Answer', 
                        alpha=0.8, color='lightcoral')
        
        plt.title(f'Strategy Usage by Answer Correctness - {clean_model_name}', 
                 fontsize=14, fontweight='bold')
        plt.xlabel('Strategy Categories', fontsize=12)
        plt.ylabel('Frequency', fontsize=12)
        plt.xticks(x, labels, rotation=45, ha='right')
        plt.legend(fontsize=11)
        plt.grid(True, alpha=0.3, axis='y')
        
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, f'strategy_correctness_{model_name}.png'), 
                    dpi=300, bbox_inches='tight')
        plt.show()
        
        # Plot 3: Effectiveness rates (separate plot)
        plt.figure(figsize=(12, 8))
        success_rates = []
        for cat in used_categories:
            correct_count = strategy_counts_correct.get(cat, 0)
            total_count = strategy_counts_all.get(cat, 0)
            rate = correct_count / total_count * 100 if total_count > 0 else 0
            success_rates.append(rate)
        
        colors = ['darkgreen' if rate >= 60 else 'gold' if rate >= 40 else 'lightcoral' 
                 for rate in success_rates]
        
        bars3 = plt.bar(range(len(used_categories)), success_rates, alpha=0.8, color=colors)
        plt.title(f'Strategy Effectiveness (% Correct Answers) - {clean_model_name}', 
                 fontsize=14, fontweight='bold')
        plt.xlabel('Strategy Categories', fontsize=12)
        plt.ylabel('Correct Answer Rate (%)', fontsize=12)
        plt.xticks(range(len(used_categories)), labels, rotation=45, ha='right')
        plt.axhline(y=50, color='red', linestyle='--', alpha=0.7, label='50% baseline')
        plt.legend(fontsize=11)
        plt.grid(True, alpha=0.3, axis='y')
        
        for bar, rate in zip(bars3, success_rates):
            plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                    f'{rate:.0f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
        
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, f'strategy_effectiveness_{model_name}.png'), 
                    dpi=300, bbox_inches='tight')
        plt.show()
        
        # Plot 4: Pie chart (separate plot)
        plt.figure(figsize=(10, 10))
        wedges, texts, autotexts = plt.pie(counts, labels=labels, autopct='%1.1f%%', 
                                          colors=plt.cm.Set3(np.linspace(0, 1, len(used_categories))))
        plt.title(f'Strategy Distribution (Proportional) - {clean_model_name}', 
                 fontsize=14, fontweight='bold')
        
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, f'strategy_pie_chart_{model_name}.png'), 
                    dpi=300, bbox_inches='tight')
        plt.show()

        # Print summary
        print(f"\n--- Official Category Summary for {clean_model_name} ---")
        print(f"Strategies mapped to official categories: {len([s for s in all_strategies if s != 'other'])}")
        print(f"Strategies categorized as 'other': {strategy_counts_all['other']}")
        
        print(f"\nOfficial category usage (ranked by frequency):")
        for i, (category, count) in enumerate(strategy_counts_all.most_common(), 1):
            if count > 0:
                success_rate = strategy_counts_correct.get(category, 0) / count * 100
                print(f"  {i:2d}. {category}: {count} uses, {success_rate:.1f}% success")

In [ ]:
if all_model_data:
    print("\n" + "="*60)
    print("RUNNING STRATEGY ANALYSIS")
    print("="*60)
    
    output_dir = os.path.join(EVAL_PATH, 'plots')
    os.makedirs(output_dir, exist_ok=True)
    
    # Run the official category analysis
    analyze_strategies_official_categories_medqa(all_model_data, output_dir)
    
    print(f"\nStrategy analysis plots saved to: {output_dir}")
elif not REASON:
    print("\nREASON=False - Strategy analysis skipped")
else:
    print("\nNo model data available for strategy analysis")

In [ ]:
# ============================================================================
# MEDQA RESPONSE ENTROPY ANALYSIS (CATEGORICAL DATA)
# ============================================================================

import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

def calculate_medqa_entropy(all_model_data, output_dir):
    """
    Calculate response entropy for MedQA data using categorical letters directly.
    Entropy is the appropriate measure for categorical data like answer choices.
    """
    results = {}
    
    for model_key, data_list in all_model_data.items():
        question_entropies = []
        question_unique_counts = []
        question_disagreement_rates = []
        
        for case in data_list:
            question_stats = case.get('question_stats')
            
            if question_stats and 'individual_responses' in question_stats:
                individual_responses = question_stats['individual_responses']
                
                if individual_responses and len(individual_responses) > 1:
                    # Get answer letters directly (no number conversion)
                    answers = []
                    for response_data in individual_responses.values():
                        answer = response_data['answer']
                        answers.append(answer)
                    
                    if len(answers) > 1:
                        # Calculate entropy directly from categorical data
                        answer_counts = Counter(answers)
                        total = len(answers)
                        entropy = 0
                        for count in answer_counts.values():
                            if count > 0:
                                prob = count / total
                                entropy -= prob * np.log2(prob)
                        question_entropies.append(entropy)
                        
                        # Additional categorical measures
                        unique_answers = len(answer_counts)
                        question_unique_counts.append(unique_answers)
                        
                        # Disagreement rate (fraction not choosing modal answer)
                        modal_count = max(answer_counts.values())
                        disagreement_rate = (total - modal_count) / total
                        question_disagreement_rates.append(disagreement_rate)
        
        if question_entropies:
            results[model_key] = {
                'mean_entropy': np.mean(question_entropies),
                'std_entropy': np.std(question_entropies),
                'mean_unique_answers': np.mean(question_unique_counts),
                'mean_disagreement_rate': np.mean(question_disagreement_rates),
                'question_entropies': question_entropies,
                'question_unique_counts': question_unique_counts,
                'question_disagreement_rates': question_disagreement_rates,
                'n_questions': len(question_entropies)
            }
    
    # Create visualizations
    if results:
        plot_medqa_entropy(results, output_dir)
    
    return results

def plot_medqa_entropy(results, output_dir):
    """Create entropy plots for MedQA categorical data"""
    os.makedirs(output_dir, exist_ok=True)
    
    model_names = list(results.keys())
    clean_labels = get_clean_labels(model_names)
    
    # Plot 1: Mean Entropy Comparison
    plt.figure(figsize=(10, 6))
    
    entropies = [results[model]['mean_entropy'] for model in model_names]
    entropy_std_errors = [results[model]['std_entropy'] / np.sqrt(results[model]['n_questions']) 
                          for model in model_names]
    
    bars = plt.bar(range(len(model_names)), entropies, yerr=entropy_std_errors, 
                   capsize=5, alpha=0.8, color=['lightsteelblue', 'lightsalmon'][:len(model_names)])
    
    plt.xlabel('Models', fontsize=12)
    plt.ylabel('Mean Response Entropy', fontsize=12)
    plt.title('MedQA: Response Entropy Comparison', 
              fontsize=14, fontweight='bold')
    plt.xticks(range(len(model_names)), clean_labels)
    plt.grid(True, alpha=0.3, axis='y')
    
    for bar, ent in zip(bars, entropies):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{ent:.3f}', ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'medqa_entropy_comparison.png'), 
                dpi=300, bbox_inches='tight')
    plt.show()
    
    # Plot 2: Mean Disagreement Rate
    plt.figure(figsize=(10, 6))
    
    disagreement_rates = [results[model]['mean_disagreement_rate'] for model in model_names]
    
    bars = plt.bar(range(len(model_names)), disagreement_rates, 
                   alpha=0.8, color=['lightcoral', 'lightblue'][:len(model_names)])
    
    plt.xlabel('Models', fontsize=12)
    plt.ylabel('Mean Disagreement Rate', fontsize=12)
    plt.title('MedQA: Response Disagreement Rate\n(Fraction Not Choosing Modal Answer)', 
              fontsize=14, fontweight='bold')
    plt.xticks(range(len(model_names)), clean_labels)
    plt.grid(True, alpha=0.3, axis='y')
    plt.ylim(0, 1)
    
    for bar, rate in zip(bars, disagreement_rates):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{rate:.3f}', ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'medqa_disagreement_rate.png'), 
                dpi=300, bbox_inches='tight')
    plt.show()
    
    # Plot 3: Entropy Distribution
    plt.figure(figsize=(10, 6))
    
    entropy_data = [results[model]['question_entropies'] for model in model_names]
    
    box_plot = plt.boxplot(entropy_data, tick_labels=clean_labels, patch_artist=True)
    plt.ylabel('Response Entropy per Question', fontsize=12)
    plt.title('MedQA: Distribution of Response Entropy Across Questions', 
              fontsize=14, fontweight='bold')
    plt.grid(True, alpha=0.3, axis='y')
    
    colors = ['lightsteelblue', 'lightsalmon']
    for patch, color in zip(box_plot['boxes'], colors[:len(model_names)]):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'medqa_entropy_distribution.png'), 
                dpi=300, bbox_inches='tight')
    plt.show()
    
    # Plot 4: Average Unique Answers
    plt.figure(figsize=(10, 6))
    
    unique_counts = [results[model]['mean_unique_answers'] for model in model_names]
    
    bars = plt.bar(range(len(model_names)), unique_counts, 
                   alpha=0.8, color=['lightgreen', 'gold'][:len(model_names)])
    
    plt.xlabel('Models', fontsize=12)
    plt.ylabel('Mean Unique Answers per Question', fontsize=12)
    plt.title('MedQA: Average Number of Different Answers Given', 
              fontsize=14, fontweight='bold')
    plt.xticks(range(len(model_names)), clean_labels)
    plt.grid(True, alpha=0.3, axis='y')
    plt.ylim(1, 5)  # Max 5 possible answers (A-E)
    
    for bar, count in zip(bars, unique_counts):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                f'{count:.2f}', ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'medqa_unique_answers.png'), 
                dpi=300, bbox_inches='tight')
    plt.show()
    
    # Print results
    print("\n" + "="*60)
    print("MEDQA ENTROPY ANALYSIS RESULTS")
    print("="*60)
    for model_name in model_names:
        clean_name = get_clean_labels([model_name])[0]
        r = results[model_name]
        print(f"{clean_name}:")
        print(f"  Mean Entropy: {r['mean_entropy']:.4f} (±{r['std_entropy']:.4f})")
        print(f"  Mean Disagreement Rate: {r['mean_disagreement_rate']:.4f}")
        print(f"  Mean Unique Answers: {r['mean_unique_answers']:.2f}")
        print(f"  Questions: {r['n_questions']}")
        print()

# Run the analysis
variance_output_dir = os.path.join(EVAL_PATH, 'plots', 'variance_analysis')
medqa_entropy_results = calculate_medqa_entropy(all_model_data, variance_output_dir)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr
import os

def scatter_and_correlation_analysis(model_data_dict):
    output_dir = 'MedQA_all/data/evaluation/plots'
    os.makedirs(output_dir, exist_ok=True)

    for model_name, data in model_data_dict.items():
        df = []
        for case in data:
            stats = case['question_stats']
            majority_answer = stats['most_frequent_answers'][0]
            row = {
                'is_correct': stats['is_correct'],
                'confidence': stats['average_uncertainty'].get(majority_answer, 0) / 100.0,
                'self_consistency': stats['self_consistency'].get(majority_answer, 0.0),
                'reasoning_token_count': len(stats.get('most_frequent_reasoning_tokens', []))
            }
            df.append(row)

        df = pd.DataFrame(df)

        for variable in ['reasoning_token_count', 'confidence', 'self_consistency']:
            x = df[variable]
            y = df['is_correct']

            # Compute correlations
            pearson_corr, _ = pearsonr(x, y)
            spearman_corr, _ = spearmanr(x, y)

            # Plot
            plt.figure(figsize=(6, 5))
            plt.scatter(x, y + 0.01 * (np.random.rand(len(y)) - 0.5), alpha=0.6, s=20)  # jitter y for better visibility
            plt.xlabel(variable.replace('_', ' ').title())
            plt.ylabel("Accuracy (1 = correct, 0 = incorrect)")
            plt.title(f"{model_name}: Accuracy vs {variable.replace('_', ' ').title()}\n"
                      f"Pearson: {pearson_corr:.2f}, Spearman: {spearman_corr:.2f}")

            filename = f"{model_name}_scatter_accuracy_vs_{variable}.png"
            plt.tight_layout()
            plt.savefig(os.path.join(output_dir, filename))
            plt.show()
